# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
import os
import sys
sys.path.append(os.path.abspath('../05_src'))
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

In [2]:
from utils.logger import get_logger
_logs = get_logger(__name__)

In [3]:
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [4]:
response = client.responses.create(
    model = 'gpt-4o-mini',
    input = 'Hello world!'
    
)

print(response.output_text)

Hello! How can I assist you today?


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [5]:
import requests
file_url = 'https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf'
book = requests.get(file_url)
book

<Response [200]>

In [6]:
dict(book.headers)

{'Date': 'Sat, 25 Apr 2026 02:13:58 GMT',
 'Server': 'Apache',
 'X-Content-Type-Options': 'nosniff',
 'Upgrade': 'h2',
 'Connection': 'Upgrade, Keep-Alive',
 'Last-Modified': 'Sun, 22 Aug 2021 17:13:44 GMT',
 'ETag': '"2d611-5ca2905de224c"',
 'Accept-Ranges': 'bytes',
 'Content-Length': '185873',
 'Cache-Control': 'max-age=31536000',
 'Expires': 'Sun, 25 Apr 2027 02:13:58 GMT',
 'Vary': 'User-Agent',
 'Keep-Alive': 'timeout=5, max=100',
 'Content-Type': 'application/pdf'}

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [7]:
system_prompt = "You are a specialist in summarizing texts for busy professionals who speaks like Gen Z."

In [8]:
import requests
from langchain_community.document_loaders import PyPDFLoader
import tempfile

url = "https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf"

pdf_bytes = requests.get(url).content

with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
    tmp.write(pdf_bytes)
    pdf_path = tmp.name

loader = PyPDFLoader(pdf_path)
pages = loader.load()


In [9]:
#joining pages
book_text = "\n\n".join([p.page_content for p in pages])

In [10]:
#Because the book is too long, I'm parsing it into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=8000,      # safe for GPT‑4o
    chunk_overlap=500
)

chunks = splitter.split_text(book_text)
len(chunks)


10

In [11]:
# after chunking, I am using the llm to summarize each chunk to generate multiple short summaries. After this, I will merge each summary to produce a big summary.
summaries = []

for chunk in chunks:
    resp = client.responses.create(
        model="gpt-4o",
        input=f"Summarize this text in 1000 tokens:\n\n{chunk}"
    )
    text = resp.output[0].content[0].text
    summaries.append(text)

In [12]:
prompt = f"""
    You are a specialist in summarizing texts. 
    Using only the text below, do the following:
    
    1. Identify the book's title and author.
    2. Summarize concisely in less than 1000 tokens the main objective of the paper.
    3. Identify and state the relevance of this book in less than 2 sentences for AI professionals and their development.
    4. State number of input tokens.
    5. State number of output tokens.
    6. State the tone of the book.

        
    The book is the following: 
    <book>
    {summaries}
    </book>

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Summary: <summary>
    Relevance: <relevance>
    Input Tokens: <input_tokens>
    Output Tokens: <output_tokens>
    Tone: <tone>

"""

In [13]:
prompt

'\n    You are a specialist in summarizing texts. \n    Using only the text below, do the following:\n\n    1. Identify the book\'s title and author.\n    2. Summarize concisely in less than 1000 tokens the main objective of the paper.\n    3. Identify and state the relevance of this book in less than 2 sentences for AI professionals and their development.\n    4. State number of input tokens.\n    5. State number of output tokens.\n    6. State the tone of the book.\n\n\n    The book is the following: \n    <book>\n    [\'Peter F. Drucker\\\'s "Managing Oneself," published in the Harvard Business Review, emphasizes the importance of self-awareness in the modern knowledge economy. He argues that with increased opportunities, individuals must become their own CEOs, taking charge of their career paths. This requires a deep understanding of one\\\'s strengths, weaknesses, values, and optimal work environment.\\n\\nDrucker advises using feedback analysis to identify strengths and improve p

In [14]:
response = client.responses.create(
    model = 'gpt-4o',
    #model = 'gpt-4o-mini', # depending on the tier we have available, we might need to update the model to be used
    input = prompt
)

In [15]:
final_summary = response.output_text

In [16]:
combined_summaries = "\n\n".join(summaries)


In [17]:
from IPython.display import display, Markdown

display(Markdown(response.output_text))

Title: Managing Oneself

Author: Peter F. Drucker

Summary: "Managing Oneself" emphasizes the vital role of self-awareness in navigating the modern knowledge economy. Drucker argues that individuals must act as their own CEOs, understanding their strengths, weaknesses, values, and ideal work environments to guide their career paths effectively. The use of feedback analysis helps identify personal strengths and areas needing improvement. By focusing on strengths and aligning them with personal values, individuals can achieve excellence and contribute significantly to organizations. Understanding how one performs, learns, and cooperates is crucial, as is setting measurable, impactful goals, and taking responsibility in professional relationships. The book also outlines the concept of pursuing a "second career" for continued engagement and fulfillment, emphasizing proactive career management.

Relevance: For AI professionals, Drucker's insights on self-management, leveraging personal strengths, and aligning career paths with values can enhance personal development and efficacy in a rapidly evolving field. Understanding different work and learning styles is particularly beneficial in collaborative AI environments.

Input Tokens: 2649

Output Tokens: 149

Tone: Informative and motivational

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
import deepeval
print(deepeval.__version__)


3.3.9


In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
...

test_case = LLMTestCase(input=combined_summaries, actual_output=final_summary)
metric = SummarizationMetric(
    threshold=0.5,
    model="gpt-3.5-turbo",
    assessment_questions=[
        "Is the coverage score based on a percentage of 'yes' answers?",
        "Does the score ensure the summary's accuracy with the source?",
        "Does a higher score mean a more comprehensive summary?"
    ]
)

# To run metric as a standalone
# metric.measure(test_case)
# print(metric.score, metric.reason)

evaluate(test_cases=[test_case], metrics=[metric])

✨ You're running DeepEval's latest Summarization Metric! (using gpt-3.5-turbo, strict=False, async_mode=True)...

ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} Retrying: 1 time(s)...


ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} Retrying: 1 time(s)...


ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} Retrying: 2 time(s)...


ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} Retrying: 2 time(s)...


...

ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} Retrying: 7 time(s)...


ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} Retrying: 8 time(s)...


ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} Retrying: 8 time(s)...


ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} Retrying: 9 time(s)...
ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} Retrying: 9 time(s)...


: 

In [1]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval

# ---------------------------------------------------------
# 1. Inputs
# ---------------------------------------------------------

ORIGINAL_TEXT = combined_summaries
SUMMARY_TEXT = final_summary

# ---------------------------------------------------------
# 2. Test Case
# ---------------------------------------------------------

test_case = LLMTestCase(
    input=ORIGINAL_TEXT,
    actual_output=SUMMARY_TEXT,
    retrieval_context=[ORIGINAL_TEXT],   # grounding for faithfulness-like checks
    context=[ORIGINAL_TEXT],             # grounding for hallucination-like checks
)

# ---------------------------------------------------------
# 3. GEval Summarization Metric
# ---------------------------------------------------------

summarization_geval = GEval(
    name="Summarization Quality",
    evaluation_steps=[
        "Evaluate whether the summary captures the key points of the source text.",
        "Check if the summary avoids adding information not present in the source.",
        "Assess whether the summary is concise yet complete.",
        "Determine whether the summary preserves the meaning and intent of the original text.",
        "Evaluate clarity and readability of the summary."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

# ---------------------------------------------------------
# 4. Run Evaluation
# ---------------------------------------------------------

results = evaluate(
    test_cases=[test_case],
    metrics=[summarization_geval]
)

results

NameError: name 'combined_summaries' is not defined

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
